# 01 — Independent Data Source Verification & Coverage Validation

**Project:** APEX — F1 Intelligence  
**Workflow Role:** Collaborative Research Workspace (Project Owner + ChatGPT)  
**Objective:** Independently audit and verify candidate Formula 1 data feeds (Jolpica, OpenF1, FastF1), compare coverage across historical eras and the current 2026 season, inspect schema fidelity and rate limits, and formulate recommendations for the initial modeling dataset.

## 1. Context & Starting Source Hypotheses

The data sources are evaluated as **starting hypotheses**, not unquestionable conclusions:
- **Hypothesis 1 (Jolpica F1 API):** Serves as an open, unauthenticated source for historical race results, qualifying classifications, standings, and pit stop records (1950–present), but lacks sub-second continuous car telemetry.
- **Hypothesis 2 (OpenF1 Community API):** Serves as an open source for modern-era session timing, car telemetry (sampled at approximately 3.7 Hz according to official documentation: https://openf1.org/docs/), tyre compound stints, weather, intervals, and race-control messages (2023–present), but lacks pre-2023 archives and enforces strict rate limits.
- **Hypothesis 3 (FastF1 Library / Upstream Timing Archive):** Serves as a valuable research package and session loader. Specific unauthenticated requests to the official live timing static archive (`livetiming.formula1.com/static/...`) return `HTTP 401 Unauthorized`; however, FastF1 remains functional when provided with valid authentication tokens (`f1auth.json`) or pre-cached local session files.

> **Principle:** Distinguish documentation claims from empirical notebook results. Record HTTP errors, missing data, and rate limits faithfully and precisely.

## 2. Jolpica F1 API Validation (Sporting Records & Historical Baseline)

> **Endpoint:** `https://api.jolpi.ca/ergast/f1/` (Ergast-compatible REST API)
> **TODO (Collaborative Step):** Test Jolpica endpoints on representative seasons (e.g., 2021, 2024, 2026 YTD). Verify race results, qualifying order, championship standings, and pit stops.

In [1]:
import urllib.request
import json
import time

JOLPICA_BASE = "https://api.jolpi.ca/ergast/f1"
HEADERS = {"User-Agent": "APEX-Notebook-Audit/1.0"}

# TODO: Query Jolpica for season schedules, round results, and qualifying
# Sample targets to test: 2021 (Abu Dhabi), 2024 (Bahrain), 2026 (YTD rounds)

def query_jolpica(path):
    url = f"{JOLPICA_BASE}/{path}"
    req = urllib.request.Request(url, headers=HEADERS)
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            lat_ms = (time.time() - t0) * 1000
            data = json.loads(resp.read().decode("utf-8"))
            return resp.status, lat_ms, data
    except Exception as e:
        lat_ms = (time.time() - t0) * 1000
        return getattr(e, "code", 0), lat_ms, str(e)

# Example check for 2024 Round 1 results:
status, lat, res = query_jolpica("2024/1/results.json")
print(f"Jolpica 2024 R1: HTTP {status} ({lat:.1f}ms)")
# TODO: Inspect payload fields (Results, Grid, Time, FastestLap, Status)

Jolpica 2024 R1: HTTP 200 (939.3ms)


## 3. OpenF1 Community API Validation (Modern Telemetry & Live Feeds)

> **Endpoint:** `https://api.openf1.org/v1/`
> **Documentation Reference:** https://openf1.org/docs/ (car telemetry sampled at ~3.7 Hz)
> **TODO (Collaborative Step):** Test OpenF1 across historical boundaries (2021, 2022, 2023, 2024, 2026). Verify availability of car telemetry, lap durations, sector splits, tyre stints, and weather.

In [3]:
OPENF1_BASE = "https://api.openf1.org/v1"

def query_openf1(endpoint, params=""):
    url = f"{OPENF1_BASE}/{endpoint}?{params}" if params else f"{OPENF1_BASE}/{endpoint}"
    req = urllib.request.Request(url, headers=HEADERS)
    t0 = time.time()
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            lat_ms = (time.time() - t0) * 1000
            data = json.loads(resp.read().decode("utf-8"))
            return resp.status, lat_ms, data
    except Exception as e:
        lat_ms = (time.time() - t0) * 1000
        return getattr(e, "code", 0), lat_ms, str(e)

# TODO: Check meeting existence for 2022 (pre-OpenF1 archive) vs 2023 (first supported season)
for yr in [2022, 2023, 2024,2025, 2026]:
    time.sleep(0.4)  # Enforce polite pacing to avoid HTTP 429
    st, lat, data = query_openf1("meetings", f"year={yr}")
    count = len(data) if isinstance(data, list) else 0
    print(f"OpenF1 Year {yr}: HTTP {st} ({count} meetings returned)")

OpenF1 Year 2022: HTTP 404 (0 meetings returned)
OpenF1 Year 2023: HTTP 200 (24 meetings returned)
OpenF1 Year 2024: HTTP 200 (25 meetings returned)
OpenF1 Year 2025: HTTP 200 (25 meetings returned)
OpenF1 Year 2026: HTTP 200 (27 meetings returned)


## 4. Dynamic Session Discovery & Detailed Telemetry Inspection (OpenF1)

> **Documentation Note:** Official OpenF1 documentation indicates that `car_data` telemetry (speed, throttle, brake, RPM, DRS, n_gear) is sampled at approximately **3.7 Hz** (Reference: https://openf1.org/docs/).
> **TODO (Collaborative Step):** Dynamically discover available sessions for a selected meeting, choose a suitable completed session (e.g., Race or Qualifying), verify returned session metadata, and then inspect child endpoints:
> - `laps` (lap duration, sector 1/2/3, pit out flag)
> - `car_data` (speed, throttle, brake, RPM, DRS, gear @ ~3.7 Hz)
> - `stints` (tyre compound, tyre age at start)
> - `intervals` (gap to leader, interval ahead)
> - `weather` (track temp, air temp, rainfall)
> - `race_control` (safety car, VSC, flag changes)

In [7]:
# Step 4.1: Dynamically discover sessions for a chosen meeting (e.g., from 2024)
time.sleep(0.4)
st_meet, _, meetings_2024 = query_openf1("meetings", "year=2024")
assert st_meet == 200 and isinstance(meetings_2024, list) and len(meetings_2024) > 0, "Failed to fetch 2024 meetings"

# Select a completed meeting (e.g., meeting 1254 - Abu Dhabi GP or the last completed round)
chosen_meeting = meetings_2024[-1]
chosen_meeting_key = chosen_meeting.get("meeting_key")
meeting_name = chosen_meeting.get("meeting_name", "Unknown Meeting")
print(f"Selected Meeting: {meeting_name} (meeting_key={chosen_meeting_key})")

# Step 4.2: Query available sessions for this meeting
time.sleep(0.4)
st_sess, _, sessions = query_openf1("sessions", f"meeting_key={chosen_meeting_key}")
assert st_sess == 200 and isinstance(sessions, list) and len(sessions) > 0, "Failed to fetch meeting sessions"

# Inspect and verify available sessions
print(f"Found {len(sessions)} sessions for meeting {chosen_meeting_key}:")
selected_session = None
for s in sessions:
    s_name = s.get("session_name", "")
    s_key = s.get("session_key")
    s_start = s.get("date_start", "")
    print(f"  Session: {s_name:<20} | Key: {s_key} | Start: {s_start}")
    if s_name == "Race" and selected_session is None:
        selected_session = s

if selected_session is None:
    selected_session = sessions[-1]

target_session_key = selected_session.get("session_key")
target_session_name = selected_session.get("session_name")
print(f"\nVerified Target Session: {target_session_name} (session_key={target_session_key})")


# Step 4.3: Discover a valid driver and audit telemetry endpoints
time.sleep(0.4)

status, latency, drivers = query_openf1(
    "drivers", f"session_key={target_session_key}"
)

if status != 200 or not isinstance(drivers, list) or not drivers:
    raise RuntimeError(
        f"Could not discover drivers: HTTP {status} | {drivers}"
    )

# Prefer driver 1 if present; otherwise use the first listed driver.
target_driver = next(
    (d for d in drivers if str(d.get("driver_number")) == "1"),
    drivers[0]
)
target_driver_number = target_driver["driver_number"]

print(
    f"Selected driver: {target_driver.get('full_name', 'Unknown')} "
    f"(#{target_driver_number})"
)

endpoints = [
    "laps",
    "car_data",
    "stints",
    "intervals",
    "weather",
    "race_control",
]

for endpoint in endpoints:
    time.sleep(0.4)

    if endpoint in {"laps", "car_data", "stints", "intervals"}:
        params = (
            f"session_key={target_session_key}"
            f"&driver_number={target_driver_number}"
        )
    else:
        params = f"session_key={target_session_key}"

    status, latency, data = query_openf1(endpoint, params)

    if isinstance(data, list):
        count = len(data)
        fields = list(data[0].keys()) if data else []
        print(
            f"{endpoint:<15} HTTP {status} | "
            f"{count:>6} records | "
            f"Fields: {fields[:8]}"
        )
    else:
        print(
            f"{endpoint:<15} HTTP {status} | "
            f"Response: {str(data)[:250]}"
        )


Selected Meeting: Abu Dhabi Grand Prix (meeting_key=1252)
Found 5 sessions for meeting 1252:
  Session: Practice 1           | Key: 9461 | Start: 2024-12-06T09:30:00+00:00
  Session: Practice 2           | Key: 9656 | Start: 2024-12-06T13:00:00+00:00
  Session: Practice 3           | Key: 9657 | Start: 2024-12-07T10:30:00+00:00
  Session: Qualifying           | Key: 9658 | Start: 2024-12-07T14:00:00+00:00
  Session: Race                 | Key: 9662 | Start: 2024-12-08T13:00:00+00:00

Verified Target Session: Race (session_key=9662)
Selected driver: Max VERSTAPPEN (#1)
laps            HTTP 200 |     58 records | Fields: ['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3']
car_data        HTTP 200 |  33619 records | Fields: ['date', 'session_key', 'throttle', 'rpm', 'brake', 'speed', 'n_gear', 'driver_number']
stints          HTTP 200 |      2 records | Fields: ['meeting_key', 'session_key', 'stint_numb

## 5. FastF1 Upstream Archive & Authentication Inspection

> **Upstream URL:** `https://livetiming.formula1.com/static/...`
> **Precision Note on Authentication:** An `HTTP 401 Unauthorized` response on a specific unauthenticated request to `livetiming.formula1.com/static/...` is evidence that direct access to that particular static archive endpoint requires credentials (such as an F1 account token via `f1auth.json`). It is not evidence that the entire FastF1 Python package or its offline/cached workflows are unusable.
> **TODO (Collaborative Step):** Test the specific static archive endpoint and record the exact HTTP response code and headers.

In [5]:
# Test a specific static archive endpoint on livetiming.formula1.com
# Note: An HTTP 401 error here indicates that this specific unauthenticated request
# is unauthorized; it is not proof that every FastF1 library feature or local cache loader is inaccessible.
F1_STATIC = "https://livetiming.formula1.com/static/2024/2024-03-02_Bahrain_Grand_Prix/2024-03-02_Race/Index.json"
req = urllib.request.Request(F1_STATIC, headers={"User-Agent": "FastF1/3.4.0 (APEX-Audit)"})
try:
    with urllib.request.urlopen(req, timeout=8) as resp:
        print("F1 Static Archive Endpoint: HTTP", resp.status)
except urllib.error.HTTPError as e:
    print(f"F1 Static Archive Endpoint: HTTP {e.code} ({e.reason}) -> Specific endpoint requires authentication.")
except Exception as e:
    print("F1 Static Archive Endpoint: Request exception:", e)

F1 Static Archive Endpoint: HTTP 401 (Unauthorized) -> Specific endpoint requires authentication.


## 6. Current 2026 Season Audit (Year-to-Date)

> **Temporal Context:** As of October 18, 2026, 21 meetings have occurred.
> **TODO (Collaborative Step):** Audit which 2026 rounds have posted results in Jolpica vs. OpenF1.

In [6]:
# TODO: Query 2026 results in Jolpica and compare with OpenF1 completed meetings
time.sleep(0.4)
st, lat, data_2026 = query_jolpica("2026/results.json?limit=1000")
if st == 200:
    races_2026 = data_2026["MRData"]["RaceTable"]["Races"]
    print(f"Jolpica 2026 Races with results: {len(races_2026)}")
    for r in races_2026:
        winner = r["Results"][0]["Driver"]["familyName"] if r.get("Results") else "None"
        r_round = r.get("round", "N/A")
        r_name = r.get("raceName", "N/A")
        r_date = r.get("date", "N/A")
        print(f"  Round {r_round}: {r_name} ({r_date}) - Winner: {winner}")

Jolpica 2026 Races with results: 5
  Round 1: Australian Grand Prix (2026-03-08) - Winner: Russell
  Round 2: Chinese Grand Prix (2026-03-15) - Winner: Antonelli
  Round 3: Japanese Grand Prix (2026-03-29) - Winner: Antonelli
  Round 4: Miami Grand Prix (2026-05-03) - Winner: Antonelli
  Round 5: Canadian Grand Prix (2026-05-24) - Winner: Antonelli


In [8]:

# Investigate 2026 Jolpica results coverage
paths_to_check = [
    ("2026 schedule", "2026.json?limit=1000"),
    ("All 2026 results", "2026/results.json?limit=1000"),
    ("Round 6 results", "2026/6/results.json"),
    ("Round 10 results", "2026/10/results.json"),
    ("Round 17 results", "2026/17/results.json"),
]

for label, path in paths_to_check:
    time.sleep(0.5)
    status, latency, payload = query_jolpica(path)

    print(f"\n{label}: HTTP {status} ({latency:.1f} ms)")

    if status != 200 or not isinstance(payload, dict):
        print("Response:", str(payload)[:300])
        continue

    metadata = payload.get("MRData", {})
    races = metadata.get("RaceTable", {}).get("Races", [])

    print(
        f"API total={metadata.get('total')}, "
        f"limit={metadata.get('limit')}, "
        f"offset={metadata.get('offset')}, "
        f"returned={len(races)}"
    )

    for race in races:
        results = race.get("Results", [])
        winner = (
            results[0].get("Driver", {}).get("familyName", "Unknown")
            if results else "No result returned"
        )
        print(
            f"  Round {race.get('round')}: "
            f"{race.get('raceName')} | {winner}"
        )



2026 schedule: HTTP 200 (823.9 ms)
API total=23, limit=100, offset=0, returned=23
  Round 1: Australian Grand Prix | No result returned
  Round 2: Chinese Grand Prix | No result returned
  Round 3: Japanese Grand Prix | No result returned
  Round 4: Miami Grand Prix | No result returned
  Round 5: Canadian Grand Prix | No result returned
  Round 6: Monaco Grand Prix | No result returned
  Round 7: Barcelona Grand Prix | No result returned
  Round 8: Austrian Grand Prix | No result returned
  Round 9: British Grand Prix | No result returned
  Round 10: Belgian Grand Prix | No result returned
  Round 11: Hungarian Grand Prix | No result returned
  Round 12: Dutch Grand Prix | No result returned
  Round 13: Italian Grand Prix | No result returned
  Round 14: Spanish Grand Prix | No result returned
  Round 15: Azerbaijan Grand Prix | No result returned
  Round 16: Bahrain Grand Prix in Malaysia | No result returned
  Round 17: Singapore Grand Prix | No result returned
  Round 18: United S

In [9]:

import time
import pandas as pd

# Obtain the 2026 calendar.
status, latency, schedule_payload = query_jolpica("2026.json?limit=100")

if status != 200 or not isinstance(schedule_payload, dict):
    raise RuntimeError(
        f"Could not load 2026 schedule: HTTP {status}"
    )

schedule = (
    schedule_payload
    .get("MRData", {})
    .get("RaceTable", {})
    .get("Races", [])
)

print(f"Calendar rounds discovered: {len(schedule)}")

# Check the result endpoint for each scheduled round.
round_records = []

for scheduled_race in schedule:
    round_number = int(scheduled_race["round"])
    time.sleep(0.4)  # Respectful pacing

    status, latency, payload = query_jolpica(
        f"2026/{round_number}/results.json?limit=100"
    )

    result_races = []
    if status == 200 and isinstance(payload, dict):
        result_races = (
            payload.get("MRData", {})
            .get("RaceTable", {})
            .get("Races", [])
        )

    race = next(
        (
            item for item in result_races
            if int(item.get("round", -1)) == round_number
        ),
        None,
    )

    results = race.get("Results", []) if race else []

    winner_record = next(
        (
            result for result in results
            if str(result.get("position")) == "1"
        ),
        None,
    )

    winner = (
        winner_record.get("Driver", {}).get("familyName")
        if winner_record else None
    )

    round_records.append({
        "round": round_number,
        "grand_prix": scheduled_race.get("raceName"),
        "scheduled_date": scheduled_race.get("date"),
        "http_status": status,
        "result_available": bool(results),
        "classified_records": len(results),
        "winner": winner,
        "latency_ms": round(latency, 1),
    })

coverage_df = pd.DataFrame(round_records)

display(coverage_df)

print("\nCoverage summary:")
print(coverage_df["result_available"].value_counts(dropna=False))


Calendar rounds discovered: 23


,round,grand_prix,scheduled_date,http_status,result_available,classified_records,winner,latency_ms
0,1,Australian Grand Prix,2026-03-08,200,True,22,Russell,1478.9
1,2,Chinese Grand Prix,2026-03-15,200,True,22,Antonelli,889.5
2,3,Japanese Grand Prix,2026-03-29,200,True,22,Antonelli,972.2
3,4,Miami Grand Prix,2026-05-03,200,True,22,Antonelli,1641.8
4,5,Canadian Grand Prix,2026-05-24,200,True,22,Antonelli,1761.7
5,6,Monaco Grand Prix,2026-06-07,200,True,22,Antonelli,1065.0
6,7,Barcelona Grand Prix,2026-06-14,200,True,22,Hamilton,1193.1
7,8,Austrian Grand Prix,2026-06-28,200,True,22,Russell,881.9
8,9,British Grand Prix,2026-07-05,200,True,22,Leclerc,998.8
9,10,Belgian Grand Prix,2026-07-19,200,True,22,Antonelli,1093.9



Coverage summary:
result_available
True     17
False     6
Name: count, dtype: int64


In [10]:

from pathlib import Path
import fastf1

# Locate the project root without assuming the notebook's working directory.
project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "AGENTS.md").is_file()
        and (path / "README.md").is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the APEX project root."
    )

# Keep FastF1's cache inside the project's ignored raw-data directory.
cache_dir = project_root / "data" / "raw" / "fastf1_cache"
cache_dir.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(cache_dir))

# One historical session, loading laps only to keep the test focused.
session = fastf1.get_session(2024, "Abu Dhabi", "R")

try:
    session.load(
        laps=True,
        telemetry=False,
        weather=False,
        messages=False,
    )

    print("FastF1 session load: SUCCESS")
    print("Event:", session.event["EventName"])
    print("Session:", session.name)
    print("Lap table shape:", session.laps.shape)
    print("Available lap columns:", list(session.laps.columns))
    print("Drivers:", session.laps["Driver"].nunique())

except Exception as exc:
    print("FastF1 session load: FAILED")
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc)[:1000])


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
core        WARNING 	Failed to load extended driver information!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
logger      WARNING 	Failed to load session status data!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
logger      WARNING 	Failed to load total lap count!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
logger      WARNING 	Failed to load track status data!
req 

FastF1 session load: SUCCESS
Event: Abu Dhabi Grand Prix
Session: Race
FastF1 session load: FAILED
Error type: DataNotLoadedError
Error: The data you are trying to access has not been loaded yet. See `Session.load`


## 7. Synthesized Coverage Matrix & Dataset Recommendation

> **TODO (Collaborative Step):** Synthesize empirical findings into the research coverage matrix.

| Dimension | Jolpica F1 | OpenF1 | FastF1 (Upstream) |
|---|---|---|---|
| **Historical Sporting (1950–2022)** | `[TODO: Confirm]` | `[TODO: Confirm]` | `[TODO: Confirm]` |
| **Modern Telemetry (2023–2025)** | `[TODO: Confirm]` | `[TODO: Confirm]` | `[TODO: Confirm]` |
| **Current Season (2026 YTD)** | `[TODO: Confirm]` | `[TODO: Confirm]` | `[TODO: Confirm]` |
| **Live Race Updates (Lap K)** | `[TODO: Confirm]` | `[TODO: Confirm]` | `[TODO: Confirm]` |
| **Authentication Barrier** | None | None (Rate limit throttled) | Specific static endpoints require F1 Account Token |

### Initial Research Dataset Recommendation:
- **Historical Baseline Dataset:** `[TODO: Define eras and rounds]`
- **Multimodal Telemetry Dataset:** `[TODO: Define eras and rounds]`
- **Initial Baseline Model (EXP-0001):** `[TODO: Confirm model and split strategy]`